[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_15/NB_bishop_ch15_em_algorithm.ipynb)

# Chapter 15 -- EM Algorithm

This notebook covers:
- K-means clustering from scratch
- Gaussian Mixture Models with EM
- Image segmentation with K-means

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.patches import Ellipse
from scipy import stats

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    for ax in fig.get_axes():
        ax.grid(False)
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

np.random.seed(42)

## 1. Synthetic Mixture Data

In [ ]:
true_means = [np.array([0, 0]), np.array([5, 5]), np.array([0, 7])]
true_covs = [
    np.array([[1.0, 0.5], [0.5, 1.0]]),
    np.array([[1.5, -0.4], [-0.4, 0.8]]),
    np.array([[0.6, 0.0], [0.0, 1.2]])]
true_weights = [0.35, 0.40, 0.25]

data = []
true_labels = []
for k, (w, mu, cov) in enumerate(zip(true_weights, true_means, true_covs)):
    n_k = int(w * 500)
    data.append(np.random.multivariate_normal(mu, cov, n_k))
    true_labels.extend([k] * n_k)

X = np.vstack(data)
true_labels = np.array(true_labels)
perm = np.random.permutation(len(X))
X, true_labels = X[perm], true_labels[perm]

colors = ['#1a3a6e', '#cd0000', '#228B22']
fig, ax = plt.subplots(figsize=(8, 6))
for k in range(3):
    mask = true_labels == k
    ax.scatter(X[mask, 0], X[mask, 1], c=colors[k], s=20, alpha=0.6, label=f'Cluster {k}')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Synthetic Mixture Data', fontweight='bold')
ax.legend()
plt.show()
print(f'Dataset: {X.shape[0]} points')

## 2. K-Means from Scratch

In [ ]:
class KMeans:
    def __init__(self, K, max_iter=100, tol=1e-6):
        self.K = K
        self.max_iter = max_iter
        self.tol = tol
        self.history = []

    def fit(self, X):
        N, D = X.shape
        self.centroids = self._kpp_init(X)
        self.history.append(self.centroids.copy())
        for it in range(self.max_iter):
            dists = np.array([np.sum((X - c)**2, axis=1) for c in self.centroids]).T
            self.labels = np.argmin(dists, axis=1)
            new_c = np.zeros_like(self.centroids)
            for k in range(self.K):
                mask = self.labels == k
                new_c[k] = X[mask].mean(axis=0) if mask.sum() > 0 else self.centroids[k]
            shift = np.linalg.norm(new_c - self.centroids)
            self.centroids = new_c
            self.history.append(self.centroids.copy())
            if shift < self.tol:
                print(f'Converged after {it+1} iterations')
                break
        self.inertia = sum(np.sum((X[self.labels == k] - self.centroids[k])**2) for k in range(self.K))
        return self

    def _kpp_init(self, X):
        centroids = [X[np.random.randint(len(X))]]
        for _ in range(1, self.K):
            dists = np.min([np.sum((X - c)**2, axis=1) for c in centroids], axis=0)
            centroids.append(X[np.random.choice(len(X), p=dists/dists.sum())])
        return np.array(centroids)

km = KMeans(K=3)
km.fit(X)
print(f'Inertia: {km.inertia:.2f}')

In [ ]:
n_show = min(6, len(km.history))
show_iters = np.linspace(0, len(km.history)-1, n_show, dtype=int)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for pi, ii in enumerate(show_iters):
    ax = axes.ravel()[pi]
    ci = km.history[ii]
    dists = np.array([np.sum((X - c)**2, axis=1) for c in ci]).T
    li = np.argmin(dists, axis=1)
    for k in range(3):
        ax.scatter(X[li==k, 0], X[li==k, 1], c=colors[k], s=15, alpha=0.4)
        ax.scatter(ci[k, 0], ci[k, 1], c=colors[k], s=200, marker='X',
                   edgecolors='white', linewidth=2, zorder=5)
    ax.set_title(f'Iteration {ii}')

fig.suptitle('K-Means Clustering Iterations', fontsize=14)
fig.tight_layout()
save_fig(fig, 'bishop_ch15_kmeans')
plt.show()

## 3. Gaussian Mixture Model with EM

In [ ]:
class GMM:
    def __init__(self, K, max_iter=100, tol=1e-6):
        self.K = K
        self.max_iter = max_iter
        self.tol = tol
        self.log_likelihoods = []
        self.history = []

    def fit(self, X):
        N, D = X.shape
        km = KMeans(self.K); km.fit(X)
        self.means = km.centroids.copy()
        self.covs = np.array([np.cov(X[km.labels==k].T) + 1e-6*np.eye(D) for k in range(self.K)])
        self.weights = np.array([(km.labels==k).mean() for k in range(self.K)])
        self.history.append((self.means.copy(), self.covs.copy(), self.weights.copy()))

        for it in range(self.max_iter):
            resp = self._e_step(X)
            self._m_step(X, resp)
            ll = self._log_likelihood(X)
            self.log_likelihoods.append(ll)
            self.history.append((self.means.copy(), self.covs.copy(), self.weights.copy()))
            if len(self.log_likelihoods) > 1 and abs(ll - self.log_likelihoods[-2]) < self.tol:
                print(f'EM converged after {it+1} iterations')
                break
        self.responsibilities = resp
        self.labels = np.argmax(resp, axis=1)
        return self

    def _e_step(self, X):
        N = X.shape[0]
        resp = np.zeros((N, self.K))
        for k in range(self.K):
            resp[:, k] = self.weights[k] * stats.multivariate_normal.pdf(X, self.means[k], self.covs[k])
        return resp / np.maximum(resp.sum(axis=1, keepdims=True), 1e-300)

    def _m_step(self, X, resp):
        N, D = X.shape
        N_k = resp.sum(axis=0)
        for k in range(self.K):
            self.means[k] = (resp[:, k] @ X) / N_k[k]
            diff = X - self.means[k]
            self.covs[k] = (diff.T @ (diff * resp[:, k:k+1])) / N_k[k] + 1e-6*np.eye(D)
            self.weights[k] = N_k[k] / N

    def _log_likelihood(self, X):
        ll = np.zeros(X.shape[0])
        for k in range(self.K):
            ll += self.weights[k] * stats.multivariate_normal.pdf(X, self.means[k], self.covs[k])
        return np.sum(np.log(np.maximum(ll, 1e-300)))

gmm = GMM(K=3)
gmm.fit(X)
print(f'Final LL: {gmm.log_likelihoods[-1]:.2f}')
print(f'Weights: {gmm.weights.round(3)}')

In [ ]:
def draw_ellipse(ax, mean, cov, color, n_std=2, alpha=0.3):
    evals, evecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(evecs[1, 0], evecs[0, 0]))
    w, h = 2 * n_std * np.sqrt(evals)
    ax.add_patch(Ellipse(xy=mean, width=w, height=h, angle=angle,
                         facecolor=color, alpha=alpha, edgecolor=color, lw=2))

n_show = min(6, len(gmm.history))
show_iters = np.linspace(0, len(gmm.history)-1, n_show, dtype=int)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for pi, ii in enumerate(show_iters):
    ax = axes.ravel()[pi]
    mi, ci, wi = gmm.history[ii]
    resp_i = np.zeros((len(X), 3))
    for k in range(3):
        resp_i[:, k] = wi[k] * stats.multivariate_normal.pdf(X, mi[k], ci[k])
    resp_i /= resp_i.sum(axis=1, keepdims=True)
    li = np.argmax(resp_i, axis=1)
    for k in range(3):
        ax.scatter(X[li==k, 0], X[li==k, 1], c=colors[k], s=15, alpha=0.4)
        draw_ellipse(ax, mi[k], ci[k], colors[k])
        ax.scatter(mi[k, 0], mi[k, 1], c=colors[k], s=100, marker='+', lw=3, zorder=5)
    ax.set_title(f'EM Iteration {ii}')
    ax.set_xlim(-4, 9); ax.set_ylim(-4, 11)

fig.suptitle('GMM EM: Iterative Convergence', fontsize=14)
fig.tight_layout()
save_fig(fig, 'bishop_ch15_gmm_em')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(gmm.log_likelihoods, 'o-', color='#1a3a6e', linewidth=2)
ax1.set_xlabel('EM Iteration'); ax1.set_ylabel('Log-Likelihood')
ax1.set_title('EM Log-Likelihood Convergence', fontweight='bold')

certainty = gmm.responsibilities.max(axis=1)
sort_idx = np.argsort(certainty)
r = gmm.responsibilities[sort_idx]
ax2.barh(range(len(X)), r[:, 0], color=colors[0])
ax2.barh(range(len(X)), r[:, 1], left=r[:, 0], color=colors[1])
ax2.barh(range(len(X)), r[:, 2], left=r[:, 0]+r[:, 1], color=colors[2])
ax2.set_xlabel('Responsibility'); ax2.set_ylabel('Point (sorted)')
ax2.set_title('GMM Soft Assignments', fontweight='bold')

fig.tight_layout()
plt.show()

for k in range(3):
    print(f'Cluster {k}: weight={gmm.weights[k]:.3f} (true {true_weights[k]:.3f}), '
          f'mean={gmm.means[k].round(2)} (true {true_means[k]})')

## 4. Image Segmentation with K-Means

In [ ]:
def create_synthetic_image(size=128):
    img = np.zeros((size, size, 3), dtype=np.float64)
    for i in range(size // 3):
        t = i / (size // 3)
        img[i, :] = [0.3 + 0.2*t, 0.5 + 0.2*t, 0.9 - 0.1*t]
    for j in range(size):
        peak = size//3 + int(15*np.sin(j*2*np.pi/size*3))
        for i in range(size//3, min(peak + size//3, size*2//3)):
            if i < size:
                img[i, j] = [0.4, 0.35, 0.3]
    for i in range(size*2//3, size):
        img[i, :] = [0.2, 0.6 + 0.1*np.random.random(), 0.15]
    cx, cy, r = size//5, size//5, size//10
    for i in range(size):
        for j in range(size):
            if (i-cx)**2 + (j-cy)**2 < r**2:
                img[i, j] = [1.0, 0.9, 0.2]
    img += np.random.normal(0, 0.03, img.shape)
    return np.clip(img, 0, 1)

img = create_synthetic_image(128)
H, W, C = img.shape
pixels = img.reshape(-1, 3)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img); axes[0].set_title('Original'); axes[0].axis('off')

for idx, K in enumerate([3, 5, 8]):
    km_img = KMeans(K=K, max_iter=50); km_img.fit(pixels)
    seg = np.clip(km_img.centroids[km_img.labels].reshape(H, W, C), 0, 1)
    axes[idx+1].imshow(seg); axes[idx+1].set_title(f'K = {K}'); axes[idx+1].axis('off')

fig.suptitle('Image Segmentation with K-Means', fontsize=14)
fig.tight_layout()
save_fig(fig, 'bishop_ch15_segmentation')
plt.show()

## 5. Elbow Method

In [ ]:
K_range = range(2, 11)
inertias = []
for K in K_range:
    km_test = KMeans(K=K, max_iter=30); km_test.fit(X)
    inertias.append(km_test.inertia)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(K_range), inertias, 'o-', color='#1a3a6e', linewidth=2, markersize=8)
ax.axvline(3, color='#cd0000', linestyle='--', alpha=0.7, label='True K=3')
ax.set_xlabel('K'); ax.set_ylabel('Inertia')
ax.set_title('Elbow Method', fontweight='bold')
ax.legend()
plt.tight_layout()
save_fig(fig, 'bishop_ch15_elbow')
plt.show()

## 6. BIC for Model Selection

In [ ]:
bics = []
for K in range(1, 8):
    g = GMM(K=K, max_iter=50); g.fit(X)
    N, D = X.shape
    n_params = K * (1 + D + D*(D+1)//2) - 1
    bic = -2 * g.log_likelihoods[-1] + n_params * np.log(N)
    bics.append(bic)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, 8), bics, 'o-', color='#228B22', linewidth=2, markersize=8)
ax.axvline(3, color='#cd0000', linestyle='--', alpha=0.7, label='True K=3')
ax.set_xlabel('Number of Components (K)'); ax.set_ylabel('BIC')
ax.set_title('BIC for GMM Model Selection', fontweight='bold')
ax.legend()
plt.tight_layout()
save_fig(fig, 'bishop_ch15_bic')
plt.show()

## Summary

- **K-Means** is the hard-assignment limit of EM for isotropic GMMs
- **EM** guarantees monotonic increase in log-likelihood
- E-step: soft assignments; M-step: parameter updates
- K-means++ initialization improves convergence
- BIC helps select the optimal number of components

In [ ]:
takeaways = [
    '1. K-means is the hard-assignment limit of EM for isotropic Gaussian mixtures.',
    '2. The EM algorithm guarantees monotonic increase in log-likelihood.',
    '3. E-step computes responsibilities; M-step updates parameters.',
    '4. K-means++ initialization significantly improves convergence.',
    '5. BIC balances fit and complexity to select optimal K.'
]
print('Key Takeaways:')
for t in takeaways:
    print(t)